## Autoencoder on initial scRNA-seq data

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import os
import glob
import functools

# Set seed
torch.manual_seed(111)
np.random.seed(111)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


We begin by loading the scRNA-seq tsv files as dataframes.

In [2]:
# Single-cell path
sc_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/scRNA-seq/"

# List files, obtain paths, then read as dfs
sc_files     = os.listdir(sc_path)
sc_filenames = ["".join([sc_path, "/" , tsv]) for tsv in sc_files]
sc_dfs       = [pd.read_table(tsv, sep = "\t", index_col = 0) for tsv in sc_filenames] # tsv

We then concatenate everything into a single dataframe, then convert to torch tensor for training.

In [5]:
# Concatenate all dataframes
sc_data = pd.concat(sc_dfs, axis = 1, join = "outer")

# Replace NaNs with 0
sc_data.fillna(0, inplace = True)

# Convert to torch tensor
sc_data = torch.tensor(sc_data.values)

# Check dims and other stats*
print(f"# Genes : {sc_data.shape[0]}")
print(f"# Cells : {sc_data.shape[1]}")

# Genes : 2008
# Cells : 60000


Convert stuff to TPMs, because this is the input our CFU regressor uses.

In [10]:
sc_data.sum(axis = 0)

tensor([7.3270e+04, 3.1026e+04, 2.5592e+04,  ..., 6.5126e+01, 6.5126e+01,
        6.5126e+01], dtype=torch.float64)

Let's build a simple autoencoder with 2 hidden layers and RELU activation all around.

In [35]:
class simpleAE(nn.Module):
    # Simple AE with 2 hidden layers
    def __init__(self, input_dim, h1, h2, latent_dim):
        super().__init__()
        # Encoder layers
        self.enc_fc1    = nn.Linear(input_dim, h1)
        self.enc_fc2    = nn.Linear(h1, h2)
        self.enc_latent = nn.Linear(h2, latent_dim)
        # Decoder layers
        self.dec_fc1    = nn.Linear(latent_dim, h2)
        self.dec_fc2    = nn.Linear(latent_dim, h2)
        self.dec_out    = nn.Linear(h1, input_dim)

    def encode(self, x):
        # Relu activation functions
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        z = self.enc_latent(h)
        return z

    def decode(self, z):
        # Relu activation functions
        h    = F.relu(self.dec_fc1(z))
        h    = F.relu(self.dec_fc2(h))
        xhat = self.dec_out(h)
        return xhat
    
    def forward(self, x):
        z    = self.encode(x)
        xhat = self.decode(z)
        return xhat, z

Now the training.

In [ ]:
def train_ae(model, X, epochs, batch_size = 200, lr = 1e-3):

    # Define dataset and batches
    dataset = TensorDataset(X)
    loader  = DataLoader(dataset, batch_size = batch_size, shuffle = True)

    # Adam optimizer with specified LR
    optim   = torch.optim.Adam(model.parameters(), lr=lr)

    # List to store loss over time
    losses  = []

    # Epochs
    for epoch in range(epochs):

        # Initialize epoch loss
        epoch_loss = 0.0

        # Batch (each is a tuple (x,) since we have no labels y)
        for batch_tuple in loader:

            # Single batch
            (xb,) = batch_tuple
            xb = xb.to(device)

            # Generate predicted x
            xhat, _ = model(xb)

            # Calculate loss
            loss = F.mse_loss(xhat, xb)

            # Clear gradient, backprop, update parmaeters
            optim.zero_grad(); loss.backward(); optim.step()

            # Total loss = average batch loss * batch size
            epoch_loss += loss.item() * xb.size(0)

        # Compute average 
        losses.append(epoch_loss / len(X))

        # Return loss every 10 
        if (epoch + 1) % 10 == 0:
            print(f"epoch {epoch+1:3d}  MSE = {losses[-1]:.4f}")
    return losses

In [ ]:
# Store the number of input dimensions
num_genes = sc_data.shape[0]

# Train the autoencoder
print("Training standard autoencoder...")

# Model
ae = simpleAE(input_dim = num_genes, 
              h1 = 500, 
              h2 = 100, 
              latent_dim = 10    # Play with latent dimension*
              ).to(device)
losses = train_ae(ae, sc_data, epochs = 200)

In [ ]:
# Extract latents
ae.eval(); ae_b.eval()
with torch.no_grad():
    z_ae   = ae.encode(counts_log.to(device)).cpu().numpy()
    z_ae_b = ae_b.encode(counts_log.to(device), batches.to(device)).cpu().numpy()

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
plot_latent(z_ae,   types,   "Plain AE — cell type", ax=axes[0,0], labels=["T","B","Mono"])
plot_latent(z_ae,   batches, "Plain AE — batch",     ax=axes[0,1], labels=["A","B"])
plot_latent(z_ae_b, types,   "Batch-aware AE — cell type", ax=axes[1,0], labels=["T","B","Mono"])
plot_latent(z_ae_b, batches, "Batch-aware AE — batch",     ax=axes[1,1], labels=["A","B"])
plt.tight_layout(); plt.show()